<center>
<img src="../../img/ods_stickers.jpg">
## ???????? ???? ?? ????????? ????????
<center>????? ?????????: ??????? ????????? ????????????? (@alexander_lazarev).

# <center>Novelty Detection ??? ????????????? ???????????</center>

????? ????? ????? ?????????? ? ????????? ?????? ?????????? ????????? ?????, ? ??? ??? ????? ????????? ????????? ?????????? ???????? ???? ? ?????????? ?????? ?????? ??????? ???? ??????, ??????? ????? ???????? ??????? ?? ??????? ? ??. ?? ????? ???? ??????? ?? ???????? ??????, ????????? ????? ???????? ?? ??????? ???? ?? ????? ???? ?????? ???????. 

? ????? ?? ????? ???????? ? ?????????? ?? ????? <strike>????-??</strike> ?????????? ?????? [??????????](https://plants-care.com) ??? ????????????? ????? ????????. ???????? ??????????? ? ????????? - ??? ?????? ? ?????????? ???????? ?????????????? ??????????? ? ??? ????????? ? ???? ?? ??? ????????? ??????. ????????, ???? ?? ??????? ?? ??????? ? ????????, ?? ??? ???????? ?????????? ?? ???? ????????? ?? ?? ????? ???????? ??? ???? ????? ??? ????????????, ??????????? ?????? ? ?????? ???????? ??, ?? ??? ???? - ???????? ??????? ?? ????????? ? ??????? ? ???????? ??????? ????????? ? ?? ?? ????? ?????? ??? ????????? ????? ???? ?? ?? ????????? ??????? ????????: ??????????? ?????????? ????????????? ???????; ???? ?????? ??? ???????? ?????? ?????? ?????????? ?????????? ???????; ???????????? ?????? ???????? ????? ? ???????, ? ??? ?????? ??????????? ?????????? ?????? ? ??????? ??? ????? ?? ??? ??????; ? ?????? ??????? ???????? ?????? ??????.

????? ????????? ???????? ??????, ??? ?????? ?? ?? - ??????????? ??????????, ??? ?????????? ? ??????????? ???????? ?? ????????? ????? ????. ????? ????????? ??????, ??? ????????? ???? ???????? ?????????? ???? ??????????? ??????????. ??? ??????????? ????????? ??????????? ? ???, ??? ?????? ????? ?? ??????????? ???????? ???? ?????? ???????????? ?????? ? ?????????????? ??? ???-?? ????? ???????? ???????? ?????????.

??????? ???????? ???????? ?????? ? ????????.

## ???????

?? ?????? ?? ??????? ????????????? ?????? Resnet50 ? ????? ?? [????-??????](https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html) ?? ??????? ? ???????? ?????? ?? [??????](https://www.kaggle.com/c/dogs-vs-cats/data). ? ????????????? ???????? ????? ?? 12500 ???????? ??????? ??????, ?? ??? ??????????? ????? 1000 (????? ?????????? ????? ???????? ??????? ????????). ?????????????? ?????? ?????????????? ? ?????? ????????? ????? ??????? [?????](https://www.dropbox.com/s/is44kutatj0e9fy/mlcourse_tutorial_data.zip?dl=0).

??? ?????????? ?? ???????? ????????? ??? ???????????: 
- Keras (Keras ?????? 1.2 ???-??? ?? ?????? ???? ? ?????? ??? Resnet50 ??? Theano)
- Theano
- sklearn
- pandas
- numpy

### ?????? ??????????? ?????????

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from keras import backend as K
from keras.applications.imagenet_utils import preprocess_input
from keras.applications.resnet50 import ResNet50
from keras.layers import Dense, Dropout, Flatten, Input
from keras.models import Model, Sequential
from keras.optimizers import SGD
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
from sklearn.cross_validation import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder

### ???????? ??????

? Keras ??? ???? ??????? ?????? ??????? ???????? ????????? ResNet50. ??? ??? ??? ????? - ??? ??????????????? ??. ???????? **include_top=False** ???????? ?? ??, ??? ??? ????????? ??????????? ??????, ?? ??? ????????? ?????. ??-?? ????, ??? ?? ????? ?????????? ?????????? ?????? ???????????? ????, ????? ?????????? ????? ????????? ???? (? ?? ???? ????????? ??? ???????? fine-tuning ??? ??? ??? ?? ???? ????? ?????? ?????????).

?????? ???????? ? ????????????? ????? ????? ??? ????? ?????? ????????? ?????????? ??????????????? Dense(2048) ????. ???? ?? ?? ?????? ??????????, ???? ???? ??? ?? ?? ????? ? ????????, ? ???????? ???? ??????? ??, ?? ?????? ?? ???????? ????? ???????? ? ?????? ????????? ??? ??????????? ???????. ??? ??? ?? ???????? ???????? ??????????? ???????? ??????????.

In [ ]:
NB_EPOCH = 20

RELEVANT_LAYER_NAME = 'relevant_layer'
IMG_SIZE = (224, 224)

NB_VAL_SAMPLES = 200
NB_TRAIN_SAMPLES = 800

TRAIN_DIR = 'data/train/'
VALID_DIR = 'data/valid/'

In [ ]:
def create_model():
        base_model = ResNet50(include_top=False, input_tensor=Input(shape=(3,) + IMG_SIZE))

        # ?????? ??? ????? ???? ?? ???????? ?????? ?? ?????????????
        for layer in base_model.layers:
            layer.trainable = False

        x = base_model.output
        x = Flatten()(x)
        x = Dropout(0.5)(x)
        # ???? ? ???????? ?? ????? ??????? ???????? ????????? ????????
        x = Dense(2048, activation='elu', name=RELEVANT_LAYER_NAME)(x)
        x = Dropout(0.5)(x)
    
        predictions = Dense(1, activation='sigmoid')(x)

        return Model(input=base_model.input, output=predictions)
    
print("Creating model..")
model = create_model()
print("Model created")

### ????-?????

In [ ]:
def apply_mean(image_data_generator):
    """Subtracts the dataset mean"""
    image_data_generator.mean = np.array([103.939, 116.779, 123.68], dtype=np.float32).reshape((3, 1, 1))

def get_train_datagen(*args, **kwargs):
    idg = ImageDataGenerator(*args, **kwargs)
    apply_mean(idg)
    return idg.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, class_mode='binary')

def get_validation_datagen():
    idg = ImageDataGenerator()
    apply_mean(idg)
    return idg.flow_from_directory(VALID_DIR, target_size=IMG_SIZE, class_mode='binary')
    
def fine_tuning(model):
    # ???????? ??? ?????????? 2 identity ????? ? 1 ?????????? 
    # (????? ?????????????????? ??????? ???????? 80 ????? ???????? ?????? ???????????)
    # ??? ???? ???? - "????????????"
    for layer in model.layers[:80]:
        layer.trainable = False
    for layer in model.layers[80:]:
        layer.trainable = True

    print("Compiling model..")
    sgd = SGD(lr=1e-4, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(optimizer=sgd, loss='binary_crossentropy', metrics=['accuracy'])
    model.fit_generator(
        get_train_datagen(rotation_range=30., shear_range=0.2, zoom_range=0.2, horizontal_flip=True),
        samples_per_epoch=NB_TRAIN_SAMPLES,
        nb_epoch=NB_EPOCH,
        validation_data=get_validation_datagen(),
        nb_val_samples=NB_VAL_SAMPLES)
    
    
fine_tuning(model)

### ?????????????? ??????????? ? ????????????? ??????

? ????? irrelevant ? ?????????? ???????????, ??????? ?????????? ?????? ?? ??????????? ? ?? ????????? ? ????? ????????. ????????? ????? ???????? ????????? ????????????? ??????? (??? ??? ?????? ?? ????????? ?? ???) ? ??????? irrelevant.

In [ ]:
def get_files(path):
    files = []
    if os.path.isdir(path):
        files = glob.glob(path + '*.jpg')
    elif path.find('*') > 0:
        files = glob.glob(path)
    else:
        files = [path]

    if not len(files):
        print('No images found by the given path')

    return files


def load_img(img_path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    return preprocess_input(x)[0]


def get_inputs(files):
    inputs = []
    for i in files:
        x = load_img(i)
        inputs.append(x)
    return inputs


relevant_files = get_files('data/valid/**/*.jpg')
print('Found {} relevant files'.format(len(relevant_files)))

irrelevant_files = get_files('data/irrelevant/*.jpg')
print('Found {} relevant files'.format(len(irrelevant_files)))

relevant_inputs = get_inputs(relevant_files)
irrelevant_inputs = get_inputs(irrelevant_files)

### ????????? ?????????

In [ ]:
def get_activation_function(m, layer):
    x = [m.layers[0].input, K.learning_phase()]
    y = [m.get_layer(layer).output]
    return K.function(x, y)


def get_activations(model, inputs, layer, class_name):
    all_activations = []
    activation_function = get_activation_function(model, layer)
    for i in range(len(inputs)):
        activations = activation_function([[inputs[i]], 0])
        all_activations.append(activations[0][0])

    df = pd.DataFrame(all_activations)
    df.insert(0, 'class', class_name)
    df.reset_index()
    return df

irrelevant_activations = get_activations(model, irrelevant_inputs, RELEVANT_LAYER_NAME, 'irrelevant')
relevant_activations = get_activations(model, relevant_inputs, RELEVANT_LAYER_NAME ,'relevant')

? ?????, ????? ??? ??????? ??????????? 2048 ????????. ??? ???????? ?? ??? ???? ??? ????????? ???????? ?????? ??????????????? ???? ???????????? ? ResNet50. ?? ???? ?? ??????? ??????, ? ????? ?? ??? ???????? ????? ??????????? ??????? ??????? ???????? ??????.

In [ ]:
irrelevant_activations.head()

In [ ]:
relevant_activations.head()

?????????? ???? - ???? ??????????? ?? ?????????? ??????? ??????? ??????????? ???????? ?????? ?? ????????.
??? ??? ???????????:
- ??? ??????????? ???????????????? ??? ?????????? ?????? ?????????? ?????????????? ???????? ?????????? ? ????????? 19%-23% ?? ?????? ??????????;
- ??? ??????????? ??????????? ? ????????????? ??????? - 20%-26%;
- ??? ????????????? ??????????? ???????? ???? 24%-28%.

#### ???????????? ???????????? ????????

(??????????? ????? ??? ???????????? ?????? VGG16 ? ???? ? 4096 ?????????)


????????? ??? ??????????? ?? ??????? ????????? ????
<img src="https://habrastorage.org/web/6c6/513/0d5/6c65130d51794868b5d14c9bf3e3b2d2.jpg"/>

????????? ??? ??????????? ?? ?????????
<img src="https://habrastorage.org/web/99a/c8e/bcf/99ac8ebcf81440e6a1be86e0497570e2.jpg"/>

????????? ??? ???????????? ???????????
<img src="https://habrastorage.org/web/7e5/c67/d08/7e5c67d08e224e7587ca74908af70a15.jpg"/>

???? ????????? ????? ???????? ??? ???? ?????????????? ? ?????? ?????????????. ??? ???? ??? ???????? ????? ????? ??????? ????????? ???????? ?? ???? ?????? ? ??? ?????? ??? ??????? ? ??????, ??? ?????? ?? ??? ????. 

????? ? ???????, ? ?????? ?? ?? ??????????? ??? ?????? ???????? ????? ??????????? ???????????? ???? ? ?????? ???????? ???????? ?????????????:

In [ ]:
def encode(df):
    label_encoder = LabelEncoder().fit(df['class'])
    labels = label_encoder.transform(df['class'])
    df = df.drop(['class'], axis=1)
    return df, labels

df = pd.concat([irrelevant_activations, relevant_activations])
X, y = encode(df)

sss = StratifiedShuffleSplit(np.zeros(y.shape[0]), test_size=0.3, random_state=23)
for train_index, test_index in sss:
    X_train, X_test = X.values[train_index], X.values[test_index]
    y_train, y_test = y[train_index], y[test_index]

In [ ]:
model = Sequential()
model.add(Dense(256, input_dim=2048, activation='elu', init='uniform'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu', init='uniform'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid', init='uniform'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(
    X_train,
    y_train,
    nb_epoch=4,
    validation_data=(X_test, y_test),
    batch_size=16)

?????! ?? ????????? ????? ????? ????? 100% ???????? ?????????????. ? ??? ???? ??????????? ?????? ????????? ???? ????? ??????? Logistic Regression?

In [ ]:
from sklearn.metrics import accuracy_score

params = {'C': [10, 2, .9, .4, .1], 'tol': [0.0001, 0.001]}
log_reg = LogisticRegression(solver='lbfgs', multi_class='multinomial', class_weight='balanced')
clf = GridSearchCV(log_reg, params, scoring='neg_log_loss', refit=True, cv=3, n_jobs=-1)
clf.fit(X_train, y_train)

print("best params: " + str(clf.best_params_))
print('best score:'+ str(clf.best_score_))

predictions = clf.predict(X_test)
print("accuracy", accuracy_score(y_test, predictions))

??? ? ??????? ? ??????? ???????? ???????? ???? ????? ??????? ????????. ?? ???????? ? ????? ???????????? LogisticRegression ??? ??? ??????????? ?????? ? ?????????????? ????????? ??????? ??????. 

<u>????? ??????, ??? ??????? ?????? ??? ????????????? ??? ????????? ?????? ??? ????? ???????????? ??????? ??????, ??? ??? ?????? ??????????? ??? ??????? ????? ????? ???? ?????.</u>

? ??????? ???????? ????????? ??? ??? ????? ???????? ? ???????????. ???????, ??? ???? ???????? ????? ??????? ? ??????????? ??? ?? ????????. ?????? ?????? ???????? ??????? ????? ??? VGG16, InceptionV3. ?????, ????????? ? ??? ?????? ?????????.